In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    # M × I (Market × Interest Rate)
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    # M × P (Market × Price)
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    # M × V (Market × Volatility)
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    # Sharpe-like ratio
    if 'return' in 'M*'.lower():
        data[f'MV_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
        main_features.append(f'MV_sharpe_M*_V*')
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    # M × S (Market × Sentiment)
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    # E × I (Economic × Interest Rate)
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    # E × P (Economic × Price)
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    # E × V (Economic × Volatility)
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    # E × S (Economic × Sentiment)
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    # I × P (Interest Rate × Price)
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    # I × V (Interest Rate × Volatility)
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    # I × S (Interest Rate × Sentiment)
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    # P × V (Price × Volatility)
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    # P × S (Price × Sentiment)
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    # Contrarian signal (opposite signs amplify)
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    # V × S (Volatility × Sentiment)
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data
    
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,0.415121,39.293238,13.819007,18.370000,0.747024,6.460530,-6.460530,8.648361,8.648361,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.720893,36.652103,17.124485,29.821070,1.341544,13.932911,-13.932911,10.385726,10.385726,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,0.813072,38.321956,22.500644,35.111530,1.485426,20.615650,-20.615650,13.878609,13.878609,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,0.707903,33.359403,25.236451,26.727384,1.382853,20.219316,-20.219316,14.621450,14.621450,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,0.671389,32.548385,21.987350,24.789544,1.295489,16.746035,-16.746035,12.926420,12.926420,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.975153,14.998988,12.413334,18.013558,1.492962,14.908227,-14.908227,9.985670,9.985670,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,1.068629,13.720627,21.347952,18.137762,1.714749,28.220582,-28.220582,16.457558,16.457558,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.779983,11.527504,17.911727,11.147219,1.459002,17.320831,-17.320831,11.871701,11.871701,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,1.205378,12.385137,7.635535,18.450361,2.161599,11.374792,-11.374792,5.262211,5.262211,0.008310


In [4]:
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_units, dropout_rate):
        super(NeuralNetwork, self).__init__()
        
        # Batch normalization for input
        self.input_bn = nn.BatchNorm1d(input_dim)
        
        # First layer with batch norm
        self.layer1 = nn.Linear(input_dim, hidden_units[0])
        self.bn1 = nn.BatchNorm1d(hidden_units[0])
        self.dropout1 = nn.Dropout(dropout_rate)
        
        # Second layer with batch norm
        self.layer2 = nn.Linear(hidden_units[0], hidden_units[1])
        self.bn2 = nn.BatchNorm1d(hidden_units[1])
        self.dropout2 = nn.Dropout(dropout_rate)
        
        # Third layer with batch norm
        self.layer3 = nn.Linear(hidden_units[1], hidden_units[2])
        self.bn3 = nn.BatchNorm1d(hidden_units[2])
        self.dropout3 = nn.Dropout(dropout_rate)
        
        # Fourth layer (added for better representation)
        self.layer4 = nn.Linear(hidden_units[2], hidden_units[3])
        self.bn4 = nn.BatchNorm1d(hidden_units[3])
        self.dropout4 = nn.Dropout(dropout_rate * 0.5)  # Less dropout in later layers
        
        # Output layer - linear activation for regression
        self.output = nn.Linear(hidden_units[3], 1)
        
        # Use ELU for better handling of negative values
        self.activation = nn.ELU()
        
        # Initialize weights properly for regression
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize weights using He initialization for better gradient flow"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Input batch normalization
        x = self.input_bn(x)
        
        # Layer 1
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.activation(x)
        x = self.dropout1(x)
        
        # Layer 2
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.activation(x)
        x = self.dropout2(x)
        
        # Layer 3
        x = self.layer3(x)
        x = self.bn3(x)
        x = self.activation(x)
        x = self.dropout3(x)
        
        # Layer 4
        x = self.layer4(x)
        x = self.bn4(x)
        x = self.activation(x)
        x = self.dropout4(x)
        
        # Output layer - no activation for regression
        x = self.output(x)
        
        return x


class PyTorchResidualEnsemble:
    
    def __init__(self, model_config, n_models=3):
        self.model_config = model_config
        self.n_models = n_models
        self.models = []
        self.scalers = []  # Store scalers for proper normalization
        self.target_scalers = []  # Store target scalers
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {self.device}')
        
    def _build_model(self, input_dim):
        """Build a neural network model"""
        model = NeuralNetwork(
            input_dim=input_dim,
            hidden_units=self.model_config['hidden_units'],
            dropout_rate=self.model_config['dropout_rate']
        ).to(self.device)
        
        return model
    
    def _train_model(self, model, train_loader, val_loader=None):
        """Train a single model with improved optimization for regression"""
        # Use AdamW optimizer with weight decay
        optimizer = optim.AdamW(
            model.parameters(),
            lr=self.model_config['learning_rate'],
            weight_decay=self.model_config['weight_decay'],
            betas=(0.9, 0.999),
            eps=1e-8
        )
        
        # Learning rate scheduler for better convergence
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=0.5, 
            patience=10, 
            verbose=self.model_config['verbose']
        )
        
        # Combined loss: MAE + MSE for better floating point prediction
        mae_criterion = nn.L1Loss()
        mse_criterion = nn.MSELoss()
        huber_criterion = nn.SmoothL1Loss()  # Robust to outliers
        
        best_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        for epoch in range(self.model_config['epochs']):
            # Training phase
            model.train()
            train_loss = 0.0
            
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                outputs = model(batch_x).squeeze()
                
                # Combined loss for better regression
                mae_loss = mae_criterion(outputs, batch_y)
                mse_loss = mse_criterion(outputs, batch_y)
                huber_loss = huber_criterion(outputs, batch_y)
                
                # Weighted combination
                loss = 0.4 * mae_loss + 0.3 * mse_loss + 0.3 * huber_loss
                
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            
            # Validation phase
            if val_loader is not None:
                model.eval()
                val_loss = 0.0
                
                with torch.no_grad():
                    for batch_x, batch_y in val_loader:
                        batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                        outputs = model(batch_x).squeeze()
                        
                        mae_loss = mae_criterion(outputs, batch_y)
                        mse_loss = mse_criterion(outputs, batch_y)
                        huber_loss = huber_criterion(outputs, batch_y)
                        loss = 0.4 * mae_loss + 0.3 * mse_loss + 0.3 * huber_loss
                        
                        val_loss += loss.item()
                
                avg_val_loss = val_loss / len(val_loader)
                monitor_loss = avg_val_loss
                
                # Update learning rate based on validation loss
                scheduler.step(avg_val_loss)
            else:
                monitor_loss = avg_train_loss
                scheduler.step(avg_train_loss)
            
            # Early stopping
            if monitor_loss < best_loss:
                best_loss = monitor_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
            
            if self.model_config['verbose'] and (epoch + 1) % 10 == 0:
                if val_loader is not None:
                    print(f'Epoch {epoch+1}/{self.model_config["epochs"]} - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}')
                else:
                    print(f'Epoch {epoch+1}/{self.model_config["epochs"]} - Train Loss: {avg_train_loss:.6f}')
            
            if patience_counter >= self.model_config['early_stopping_patience']:
                print(f'Early stopping at epoch {epoch+1}')
                break
        
        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        return model
    
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        # Convert to numpy arrays if not already
        train_x = np.array(train_x, dtype=np.float64)
        train_y = np.array(train_y, dtype=np.float64)
        
        current_train_pred = np.zeros(len(train_y), dtype=np.float64)
        
        for i in range(self.n_models):
            print(f'\n{"="*50}')
            print(f'Training Model {i+1}/{self.n_models}')
            print("="*50)
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            # Scale features for better gradient flow
            scaler = StandardScaler()
            train_x_scaled = scaler.fit_transform(train_x)
            self.scalers.append(scaler)
            
            # Scale target for better convergence
            target_scaler = StandardScaler()
            target_scaled = target_scaler.fit_transform(target.reshape(-1, 1)).flatten()
            self.target_scalers.append(target_scaler)
            
            # Create PyTorch datasets with scaled data
            train_dataset = TensorDataset(
                torch.FloatTensor(train_x_scaled),
                torch.FloatTensor(target_scaled)
            )
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.model_config['batch_size'],
                shuffle=True
            )
            
            # Create validation loader if test data provided
            val_loader = None
            if test_x is not None and test_y is not None:
                test_x_np = np.array(test_x, dtype=np.float64)
                test_y_np = np.array(test_y, dtype=np.float64)
                
                if i == 0:
                    val_target = test_y_np
                else:
                    current_test_pred = self.predict(test_x_np)
                    val_target = test_y_np - current_test_pred
                
                test_x_scaled = scaler.transform(test_x_np)
                val_target_scaled = target_scaler.transform(val_target.reshape(-1, 1)).flatten()
                
                val_dataset = TensorDataset(
                    torch.FloatTensor(test_x_scaled),
                    torch.FloatTensor(val_target_scaled)
                )
                val_loader = DataLoader(
                    val_dataset,
                    batch_size=self.model_config['batch_size'],
                    shuffle=False
                )
            
            # Build and train model
            model = self._build_model(train_x.shape[1])
            model = self._train_model(model, train_loader, val_loader)
            self.models.append(model)
            
            # Update cumulative predictions (unscaled)
            pred = self._predict_single(model, train_x_scaled, target_scaler)
            current_train_pred += pred
            
            train_mae = mean_absolute_error(train_y, current_train_pred)
            train_rmse = np.sqrt(mean_squared_error(train_y, current_train_pred))
            print(f'\nCumulative Training MAE: {train_mae:.6f}')
            print(f'Cumulative Training RMSE: {train_rmse:.6f}')
        
        return self
    
    def _predict_single(self, model, X_scaled, target_scaler):
        """Predict with a single model"""
        model.eval()
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        
        with torch.no_grad():
            predictions_scaled = model(X_tensor).cpu().numpy().flatten()
        
        # Inverse transform to get original scale
        predictions = target_scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()
        
        return predictions
    
    def predict(self, X):
        X = np.array(X, dtype=np.float64)
        predictions = np.zeros(len(X), dtype=np.float64)
        
        for i, (model, scaler, target_scaler) in enumerate(zip(self.models, self.scalers, self.target_scalers)):
            X_scaled = scaler.transform(X)
            pred = self._predict_single(model, X_scaled, target_scaler)
            predictions += pred
        
        return predictions
    
    def save(self, filepath):
        """Save the ensemble model"""
        save_data = {
            'model_config': self.model_config,
            'n_models': self.n_models,
            'input_dim': self.models[0].layer1.in_features,
            'scalers': self.scalers,
            'target_scalers': self.target_scalers
        }
        
        # Save each PyTorch model separately
        for i, model in enumerate(self.models):
            torch.save(model.state_dict(), f'{filepath}_model_{i}.pt')
        
        # Save metadata and scalers
        joblib.dump(save_data, f'{filepath}_metadata.pkl')
        print(f'\nEnsemble saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        """Load the ensemble model"""
        # Load metadata
        save_data = joblib.load(f'{filepath}_metadata.pkl')
        
        # Recreate ensemble
        ensemble = PyTorchResidualEnsemble(
            model_config=save_data['model_config'],
            n_models=save_data['n_models']
        )
        
        ensemble.scalers = save_data['scalers']
        ensemble.target_scalers = save_data['target_scalers']
        
        # Load each PyTorch model
        for i in range(save_data['n_models']):
            model = NeuralNetwork(
                input_dim=save_data['input_dim'],
                hidden_units=save_data['model_config']['hidden_units'],
                dropout_rate=save_data['model_config']['dropout_rate']
            ).to(ensemble.device)
            
            model.load_state_dict(torch.load(f'{filepath}_model_{i}.pt'))
            model.eval()
            ensemble.models.append(model)
        
        return ensemble
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        # Additional metrics for regression
        r2_score = 1 - (np.sum((y - predictions) ** 2) / np.sum((y - np.mean(y)) ** 2))
        
        print(f'\n{"="*50}')
        print('Evaluation Metrics:')
        print("="*50)
        print(f'MAPE: {mape:.6f} ({mape*100:.4f}%)')
        print(f'MAE: {mae:.6f}')
        print(f'RMSE: {rmse:.6f}')
        print(f'R² Score: {r2_score:.6f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse, 'r2': r2_score}


# Enhanced PyTorch Configuration for Regression
PYTORCH_CONFIG = {
    'hidden_units': [512, 256, 128, 64],  # Deeper network with 4 layers
    'learning_rate': 0.001,
    'dropout_rate': 0.3,
    'weight_decay': 0.001,  # L2 regularization
    'epochs': 300,
    'batch_size': 64,
    'early_stopping_patience': 30,
    'verbose': True
}

# Train the ensemble
ensemble = PyTorchResidualEnsemble(model_config=PYTORCH_CONFIG, n_models=3)
ensemble.fit(train_x, train_y, test_x=test_x, test_y=test_y)  # Optional test data

# Save the model
ensemble.save('PyTorch_Residual_Ensemble')

# Evaluate
print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

if test_x is not None and test_y is not None:
    print('\nFinal Test Evaluation:')
    ensemble.evaluate(test_x, test_y)

# Load the model (when needed)
loaded_model = PyTorchResidualEnsemble.load('PyTorch_Residual_Ensemble')
# predictions = loaded_model.predict(test_x)

Using device: cpu

Training Model 1/3
Epoch 10/300 - Train Loss: 0.728354 - Val Loss: 0.649471
Epoch 20/300 - Train Loss: 0.696815 - Val Loss: 0.637403
Epoch 30/300 - Train Loss: 0.666234 - Val Loss: 0.649539
Epoch 40/300 - Train Loss: 0.640225 - Val Loss: 0.669691
Early stopping at epoch 41

Cumulative Training MAE: 0.007000
Cumulative Training RMSE: 0.009905

Training Model 2/3
Epoch 10/300 - Train Loss: 0.756285 - Val Loss: 0.805774
Epoch 20/300 - Train Loss: 0.715259 - Val Loss: 0.772257
Epoch 30/300 - Train Loss: 0.707028 - Val Loss: 0.772490
Early stopping at epoch 38

Cumulative Training MAE: 0.006851
Cumulative Training RMSE: 0.009721

Training Model 3/3
Epoch 10/300 - Train Loss: 0.737146 - Val Loss: 0.808540
Epoch 20/300 - Train Loss: 0.708214 - Val Loss: 0.798667
Epoch 30/300 - Train Loss: 0.706095 - Val Loss: 0.802240
Early stopping at epoch 34

Cumulative Training MAE: 0.006788
Cumulative Training RMSE: 0.009622

Ensemble saved to PyTorch_Residual_Ensemble

Final Training 

In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        #print(list(test_df.to_numpy()))
        test_processed = preprocessing(test_df, 'inference')
        
        LGBM_R_y_pred = loaded_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        print(float(signal))
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

1.566105694451835
0.4744976385845803
2.0
1.9099038201384246
1.793011992936954
1.627171341329813
1.1005734477075748
1.1061151619069278
2.0
1.3675601416034624
